## 1. Import Libraries and Load Data

In [76]:
import json
import numpy as np
from collections import defaultdict
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage

In [77]:
with open('actual_pr_reviewer_assignments.json', 'r', encoding='utf-8') as f:
    actual_data = json.load(f)

with open('all_pr_reviewer_assignments.json', 'r', encoding='utf-8') as f:
    predicted_data = json.load(f)

print(f"Total PRs in actual data: {len(actual_data['pr_reviewer_assignments'])}")
print(f"Total PRs in predicted data: {len(predicted_data['pr_assignments'])}")

Total PRs in actual data: 35
Total PRs in predicted data: 35


## 2. Data Preparation

In [78]:
actual_reviewers = {}
predicted_reviewers = {}

for pr in actual_data['pr_reviewer_assignments']:
    pr_number = pr['pr_number']
    actual_reviewers[pr_number] = pr['unique_reviewers']

for pr in predicted_data['pr_assignments']:
    pr_number = pr['pr_number']
    
    # Extract top-3 reviewer names from the detailed predictions
    if 'top_3_reviewers' in pr and pr['top_3_reviewers']:
        top_3_names = [reviewer['reviewer_name'] for reviewer in pr['top_3_reviewers']]
        predicted_reviewers[pr_number] = top_3_names
    else:
        # Fallback to assigned_reviewer if top_3_reviewers is not available
        predicted_reviewers[pr_number] = [pr['assigned_reviewer']] if pr.get('assigned_reviewer') else []

common_prs = set(actual_reviewers.keys()) & set(predicted_reviewers.keys())


## 3. Comprehensive Metric Calculation Functions

In [79]:
def calculate_top_k_accuracy(actual, predicted, k=1):
    if not common_prs:
        return 0.0
    
    hits = 0
    total = 0
    
    for pr_num in common_prs:
        actual_revs = set(actual[pr_num])
        pred_revs = predicted[pr_num][:k]
        
        if any(rev in actual_revs for rev in pred_revs):
            hits += 1
        total += 1
    
    return (hits / total) * 100 if total > 0 else 0.0

In [80]:
# ==================== PRECISION@K ====================
def calculate_precision_at_k(actual, predicted, k):
    """
    Precision@k = (# of recommended items @k that are relevant) / k
    """
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    if k == 0:
        return 0.0
    
    relevant_at_k = sum(1 for p in pred_k if p in actual_set)
    return relevant_at_k / k

def calculate_avg_precision_at_k(actual_dict, predicted_dict, k):
    """Calculate average Precision@k across all PRs"""
    if not common_prs:
        return 0.0
    
    precision_scores = []
    for pr_num in common_prs:
        precision = calculate_precision_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        precision_scores.append(precision)
    
    return np.mean(precision_scores) * 100

In [81]:
# ==================== RECALL@K ====================
def calculate_recall_at_k(actual, predicted, k):
    """
    Recall@k = (# of recommended items @k that are relevant) / (total # of relevant items)
    """
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    if len(actual_set) == 0:
        return 0.0
    
    relevant_at_k = sum(1 for p in pred_k if p in actual_set)
    return relevant_at_k / len(actual_set)

def calculate_avg_recall_at_k(actual_dict, predicted_dict, k):
    """Calculate average Recall@k across all PRs"""
    if not common_prs:
        return 0.0
    
    recall_scores = []
    for pr_num in common_prs:
        recall = calculate_recall_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        recall_scores.append(recall)
    
    return np.mean(recall_scores) * 100

In [82]:
# ==================== F1-SCORE@K ====================
def calculate_f1_at_k(actual, predicted, k):
    """
    F1@k = 2 * (Precision@k * Recall@k) / (Precision@k + Recall@k)
    """
    precision = calculate_precision_at_k(actual, predicted, k)
    recall = calculate_recall_at_k(actual, predicted, k)
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * (precision * recall) / (precision + recall)

def calculate_avg_f1_at_k(actual_dict, predicted_dict, k):
    """Calculate average F1@k across all PRs"""
    if not common_prs:
        return 0.0
    
    f1_scores = []
    for pr_num in common_prs:
        f1 = calculate_f1_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        f1_scores.append(f1)
    
    return np.mean(f1_scores) * 100

In [83]:
# ==================== HIT@K ====================
def calculate_hit_at_k(actual, predicted, k):
    """
    Hit@k = 1 if at least one relevant item is in top-k, else 0
    """
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    return 1.0 if any(p in actual_set for p in pred_k) else 0.0

def calculate_avg_hit_at_k(actual_dict, predicted_dict, k):
    """Calculate average Hit@k across all PRs (same as Top-k Accuracy)"""
    if not common_prs:
        return 0.0
    
    hit_scores = []
    for pr_num in common_prs:
        hit = calculate_hit_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        hit_scores.append(hit)
    
    return np.mean(hit_scores) * 100

In [84]:
def calculate_average_precision_at_k(actual, predicted, k=5):
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    if len(actual_set) == 0:
        return 0.0
    
    score = 0.0
    num_hits = 0.0
    
    for i, p in enumerate(pred_k):
        if p in actual_set:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    
    return score / min(len(actual_set), k)

def calculate_map_at_k(actual, predicted, k=5):
    if not common_prs:
        return 0.0
    
    ap_scores = []
    
    for pr_num in common_prs:
        ap = calculate_average_precision_at_k(
            actual[pr_num], 
            predicted[pr_num], 
            k
        )
        ap_scores.append(ap)
    
    return np.mean(ap_scores) * 100

In [85]:
def calculate_reciprocal_rank(actual, predicted):
    actual_set = set(actual)
    
    for i, p in enumerate(predicted):
        if p in actual_set:
            return 1.0 / (i + 1.0)
    
    return 0.0

def calculate_mrr(actual, predicted):
    if not common_prs:
        return 0.0
    
    rr_scores = []
    
    for pr_num in common_prs:
        rr = calculate_reciprocal_rank(
            actual[pr_num], 
            predicted[pr_num]
        )
        rr_scores.append(rr)
    
    return np.mean(rr_scores)

In [86]:
# ==================== DCG & NDCG ====================
def calculate_dcg_at_k(actual, predicted, k):
    """
    DCG@k = sum(rel_i / log2(i + 1)) for i in range(k)
    rel_i = 1 if predicted[i] is relevant, else 0
    """
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    dcg = 0.0
    for i, p in enumerate(pred_k):
        if p in actual_set:
            # relevance = 1 for relevant items
            dcg += 1.0 / np.log2(i + 2)  # i+2 because log2(1) = 0
    
    return dcg

def calculate_idcg_at_k(actual, k):
    """
    IDCG@k = DCG of ideal ranking (all relevant items first)
    """
    num_relevant = min(len(actual), k)
    idcg = 0.0
    for i in range(num_relevant):
        idcg += 1.0 / np.log2(i + 2)
    
    return idcg

def calculate_ndcg_at_k(actual, predicted, k):
    """
    NDCG@k = DCG@k / IDCG@k
    """
    dcg = calculate_dcg_at_k(actual, predicted, k)
    idcg = calculate_idcg_at_k(actual, k)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg

def calculate_avg_dcg_at_k(actual_dict, predicted_dict, k):
    """Calculate average DCG@k across all PRs"""
    if not common_prs:
        return 0.0
    
    dcg_scores = []
    for pr_num in common_prs:
        dcg = calculate_dcg_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        dcg_scores.append(dcg)
    
    return np.mean(dcg_scores)

def calculate_avg_ndcg_at_k(actual_dict, predicted_dict, k):
    """Calculate average NDCG@k across all PRs"""
    if not common_prs:
        return 0.0
    
    ndcg_scores = []
    for pr_num in common_prs:
        ndcg = calculate_ndcg_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        ndcg_scores.append(ndcg)
    
    return np.mean(ndcg_scores) * 100

In [87]:
def calculate_stability_metrics(actual, predicted):
    # Get all unique reviewers
    all_actual_reviewers = set()
    all_predicted_reviewers = set()
    
    for pr_num in common_prs:
        all_actual_reviewers.update(actual[pr_num])
        all_predicted_reviewers.update(predicted[pr_num])
    
    # Coverage: How many actual reviewers were predicted at least once
    coverage = len(all_actual_reviewers & all_predicted_reviewers) / len(all_actual_reviewers) if all_actual_reviewers else 0
    
    # Diversity ratio
    diversity_ratio = len(all_predicted_reviewers) / len(all_actual_reviewers) if all_actual_reviewers else 0
    
    # Exact match rate (predicted reviewer exactly matches one of the actual reviewers)
    exact_matches = sum(1 for pr_num in common_prs if set(predicted[pr_num]) & set(actual[pr_num]))
    exact_match_rate = (exact_matches / len(common_prs)) * 100 if common_prs else 0
    
    return {
        'coverage': coverage * 100,
        'diversity_ratio': diversity_ratio,
        'exact_match_rate': exact_match_rate,
        'unique_actual_reviewers': len(all_actual_reviewers),
        'unique_predicted_reviewers': len(all_predicted_reviewers)
    }

## 4. Calculate All Metrics

In [88]:
# ==================== CALCULATE ALL METRICS ====================

# Precision@k
precision_at_1 = calculate_avg_precision_at_k(actual_reviewers, predicted_reviewers, k=1)
precision_at_3 = calculate_avg_precision_at_k(actual_reviewers, predicted_reviewers, k=3)
precision_at_5 = calculate_avg_precision_at_k(actual_reviewers, predicted_reviewers, k=5)
precision_at_10 = calculate_avg_precision_at_k(actual_reviewers, predicted_reviewers, k=10)

# Recall@k
recall_at_1 = calculate_avg_recall_at_k(actual_reviewers, predicted_reviewers, k=1)
recall_at_3 = calculate_avg_recall_at_k(actual_reviewers, predicted_reviewers, k=3)
recall_at_5 = calculate_avg_recall_at_k(actual_reviewers, predicted_reviewers, k=5)
recall_at_10 = calculate_avg_recall_at_k(actual_reviewers, predicted_reviewers, k=10)

# F1-Score@k
f1_at_1 = calculate_avg_f1_at_k(actual_reviewers, predicted_reviewers, k=1)
f1_at_3 = calculate_avg_f1_at_k(actual_reviewers, predicted_reviewers, k=3)
f1_at_5 = calculate_avg_f1_at_k(actual_reviewers, predicted_reviewers, k=5)
f1_at_10 = calculate_avg_f1_at_k(actual_reviewers, predicted_reviewers, k=10)

# Hit@k (Top-k Accuracy)
hit_at_1 = calculate_avg_hit_at_k(actual_reviewers, predicted_reviewers, k=1)
hit_at_3 = calculate_avg_hit_at_k(actual_reviewers, predicted_reviewers, k=3)
hit_at_5 = calculate_avg_hit_at_k(actual_reviewers, predicted_reviewers, k=5)
hit_at_10 = calculate_avg_hit_at_k(actual_reviewers, predicted_reviewers, k=10)

# MRR (Mean Reciprocal Rank)
mrr = calculate_mrr(actual_reviewers, predicted_reviewers)

# MAP@k (Mean Average Precision)
map_at_1 = calculate_map_at_k(actual_reviewers, predicted_reviewers, k=1)
map_at_3 = calculate_map_at_k(actual_reviewers, predicted_reviewers, k=3)
map_at_5 = calculate_map_at_k(actual_reviewers, predicted_reviewers, k=5)
map_at_10 = calculate_map_at_k(actual_reviewers, predicted_reviewers, k=10)

# DCG@k (Discounted Cumulative Gain)
dcg_at_1 = calculate_avg_dcg_at_k(actual_reviewers, predicted_reviewers, k=1)
dcg_at_3 = calculate_avg_dcg_at_k(actual_reviewers, predicted_reviewers, k=3)
dcg_at_5 = calculate_avg_dcg_at_k(actual_reviewers, predicted_reviewers, k=5)
dcg_at_10 = calculate_avg_dcg_at_k(actual_reviewers, predicted_reviewers, k=10)

# NDCG@k (Normalized Discounted Cumulative Gain)
ndcg_at_1 = calculate_avg_ndcg_at_k(actual_reviewers, predicted_reviewers, k=1)
ndcg_at_3 = calculate_avg_ndcg_at_k(actual_reviewers, predicted_reviewers, k=3)
ndcg_at_5 = calculate_avg_ndcg_at_k(actual_reviewers, predicted_reviewers, k=5)
ndcg_at_10 = calculate_avg_ndcg_at_k(actual_reviewers, predicted_reviewers, k=10)

# Stability Metrics
stability = calculate_stability_metrics(actual_reviewers, predicted_reviewers)

print("✅ All metrics calculated successfully!")

✅ All metrics calculated successfully!


In [89]:
print("="*80)
print("PR REVIEWER ASSIGNMENT - COMPREHENSIVE EVALUATION METRICS")
print("="*80)
print(f"\nTotal PRs Evaluated: {len(common_prs)}")
print("\n" + "="*80)

# ==================== PRECISION@K ====================
print("\n📊 PRECISION@K")
print("-"*80)
print(f"Precision@1:  {precision_at_1:.2f}%")
print(f"Precision@3:  {precision_at_3:.2f}%")
print(f"Precision@5:  {precision_at_5:.2f}%")
print(f"Precision@10: {precision_at_10:.2f}%")

# ==================== RECALL@K ====================
print("\n📊 RECALL@K")
print("-"*80)
print(f"Recall@1:  {recall_at_1:.2f}%")
print(f"Recall@3:  {recall_at_3:.2f}%")
print(f"Recall@5:  {recall_at_5:.2f}%")
print(f"Recall@10: {recall_at_10:.2f}%")

# ==================== F1-SCORE@K ====================
print("\n📊 F1-SCORE@K")
print("-"*80)
print(f"F1@1:  {f1_at_1:.2f}%")
print(f"F1@3:  {f1_at_3:.2f}%")
print(f"F1@5:  {f1_at_5:.2f}%")
print(f"F1@10: {f1_at_10:.2f}%")

# ==================== HIT@K (Top-k Accuracy) ====================
print("\n📊 HIT@K (Top-k Accuracy)")
print("-"*80)
print(f"Hit@1:  {hit_at_1:.2f}%")
print(f"Hit@3:  {hit_at_3:.2f}%")
print(f"Hit@5:  {hit_at_5:.2f}%")
print(f"Hit@10: {hit_at_10:.2f}%")

# ==================== MRR ====================
print("\n📊 MEAN RECIPROCAL RANK (MRR)")
print("-"*80)
print(f"MRR:       {mrr:.4f}")
print(f"MRR (%):   {mrr*100:.2f}%")

# ==================== MAP@K ====================
print("\n📊 MEAN AVERAGE PRECISION (MAP@K)")
print("-"*80)
print(f"MAP@1:  {map_at_1:.2f}%")
print(f"MAP@3:  {map_at_3:.2f}%")
print(f"MAP@5:  {map_at_5:.2f}%")
print(f"MAP@10: {map_at_10:.2f}%")

# ==================== DCG@K ====================
print("\n📊 DISCOUNTED CUMULATIVE GAIN (DCG@K)")
print("-"*80)
print(f"DCG@1:  {dcg_at_1:.4f}")
print(f"DCG@3:  {dcg_at_3:.4f}")
print(f"DCG@5:  {dcg_at_5:.4f}")
print(f"DCG@10: {dcg_at_10:.4f}")

# ==================== NDCG@K ====================
print("\n📊 NORMALIZED DISCOUNTED CUMULATIVE GAIN (NDCG@K)")
print("-"*80)
print(f"NDCG@1:  {ndcg_at_1:.2f}%")
print(f"NDCG@3:  {ndcg_at_3:.2f}%")
print(f"NDCG@5:  {ndcg_at_5:.2f}%")
print(f"NDCG@10: {ndcg_at_10:.2f}%")

# ==================== STABILITY METRICS ====================
print("\n📊 STABILITY METRICS")
print("-"*80)
print(f"Coverage (Actual reviewers in predictions): {stability['coverage']:.2f}%")
print(f"Exact Match Rate:                           {stability['exact_match_rate']:.2f}%")
print(f"Diversity Ratio:                            {stability['diversity_ratio']:.2f}")
print(f"Unique Actual Reviewers:                    {stability['unique_actual_reviewers']}")
print(f"Unique Predicted Reviewers:                 {stability['unique_predicted_reviewers']}")

print("\n" + "="*80)
print("✅ Evaluation Complete!")
print("="*80)

PR REVIEWER ASSIGNMENT - COMPREHENSIVE EVALUATION METRICS

Total PRs Evaluated: 35


📊 PRECISION@K
--------------------------------------------------------------------------------
Precision@1:  11.43%
Precision@3:  16.19%
Precision@5:  9.71%
Precision@10: 4.86%

📊 RECALL@K
--------------------------------------------------------------------------------
Recall@1:  9.14%
Recall@3:  38.29%
Recall@5:  38.29%
Recall@10: 38.29%

📊 F1-SCORE@K
--------------------------------------------------------------------------------
F1@1:  9.52%
F1@3:  21.71%
F1@5:  14.88%
F1@10: 8.38%

📊 HIT@K (Top-k Accuracy)
--------------------------------------------------------------------------------
Hit@1:  11.43%
Hit@3:  45.71%
Hit@5:  45.71%
Hit@10: 45.71%

📊 MEAN RECIPROCAL RANK (MRR)
--------------------------------------------------------------------------------
MRR:       0.2524
MRR (%):   25.24%

📊 MEAN AVERAGE PRECISION (MAP@K)
-----------------------------------------------------------------------------

In [90]:
# ==================== METRICS SUMMARY TABLE ====================
import pandas as pd

metrics_summary = pd.DataFrame({
    'k': [1, 3, 5, 10],
    'Precision@k': [precision_at_1, precision_at_3, precision_at_5, precision_at_10],
    'Recall@k': [recall_at_1, recall_at_3, recall_at_5, recall_at_10],
    'F1@k': [f1_at_1, f1_at_3, f1_at_5, f1_at_10],
    'Hit@k': [hit_at_1, hit_at_3, hit_at_5, hit_at_10],
    'MAP@k': [map_at_1, map_at_3, map_at_5, map_at_10],
    'NDCG@k': [ndcg_at_1, ndcg_at_3, ndcg_at_5, ndcg_at_10]
})

print("\n" + "="*80)
print("📊 METRICS SUMMARY TABLE")
print("="*80)
print(metrics_summary.to_string(index=False))
print("\n" + "="*80)


📊 METRICS SUMMARY TABLE
 k  Precision@k  Recall@k      F1@k     Hit@k     MAP@k    NDCG@k
 1    11.428571  9.142857  9.523810 11.428571 11.428571 11.428571
 3    16.190476 38.285714 21.714286 45.714286 21.666667 27.126720
 5     9.714286 38.285714 14.884354 45.714286 20.904762 26.520394
10     4.857143 38.285714  8.380952 45.714286 20.904762 26.520394



In [91]:
# ==================== ANALYZE MULTIPLE REVIEWERS HANDLING ====================
print("🔍 ANALYZING MULTIPLE REVIEWERS IN EVALUATION")
print("=" * 80)

# Analyze the distribution of actual reviewers per PR
reviewer_count_distribution = {}
multiple_reviewer_examples = []

for pr_num in common_prs:
    actual_revs = actual_reviewers[pr_num]
    num_reviewers = len(actual_revs)
    
    # Count distribution
    reviewer_count_distribution[num_reviewers] = reviewer_count_distribution.get(num_reviewers, 0) + 1
    
    # Collect examples of PRs with multiple reviewers
    if num_reviewers > 1:
        multiple_reviewer_examples.append({
            'pr_number': pr_num,
            'actual_reviewers': actual_revs,
            'predicted_top3': predicted_reviewers[pr_num][:3],
            'num_actual': num_reviewers
        })

print(f"📊 REVIEWER COUNT DISTRIBUTION:")
total_prs = len(common_prs)
for count in sorted(reviewer_count_distribution.keys()):
    prs = reviewer_count_distribution[count]
    percentage = (prs / total_prs) * 100
    print(f"  {count} reviewer(s): {prs:2d} PRs ({percentage:4.1f}%)")

print(f"\n🎯 PRs WITH MULTIPLE REVIEWERS:")
print(f"Total PRs with multiple reviewers: {len(multiple_reviewer_examples)}")
print(f"Percentage of multi-reviewer PRs: {(len(multiple_reviewer_examples)/total_prs)*100:.1f}%")

# Show examples of how evaluation handles multiple reviewers
print(f"\n📋 EXAMPLES OF MULTIPLE REVIEWER EVALUATION:")
for i, example in enumerate(multiple_reviewer_examples[:5], 1):
    pr_num = example['pr_number']
    actual = set(example['actual_reviewers'])
    predicted = example['predicted_top3']
    
    # Calculate metrics for this specific PR
    matches_in_top3 = [p for p in predicted if p in actual]
    
    print(f"\n  Example {i}: PR #{pr_num}")
    print(f"    Actual reviewers ({len(actual)}): {', '.join(actual)}")
    print(f"    Predicted top-3: {', '.join(predicted)}")
    print(f"    Matches found: {', '.join(matches_in_top3) if matches_in_top3 else 'None'}")
    print(f"    Hit@3: {'✅ YES' if matches_in_top3 else '❌ NO'}")
    
    # Show how precision/recall calculated for this PR
    if len(predicted) > 0:
        example_precision_at_3 = len(matches_in_top3) / min(3, len(predicted))
        example_recall_at_3 = len(matches_in_top3) / len(actual)
        print(f"    Precision@3: {example_precision_at_3:.3f} ({len(matches_in_top3)}/{min(3, len(predicted))})")
        print(f"    Recall@3: {example_recall_at_3:.3f} ({len(matches_in_top3)}/{len(actual)})")

print("\n" + "=" * 80)

🔍 ANALYZING MULTIPLE REVIEWERS IN EVALUATION
📊 REVIEWER COUNT DISTRIBUTION:
  1 reviewer(s): 24 PRs (68.6%)
  2 reviewer(s):  6 PRs (17.1%)
  3 reviewer(s):  4 PRs (11.4%)
  5 reviewer(s):  1 PRs ( 2.9%)

🎯 PRs WITH MULTIPLE REVIEWERS:
Total PRs with multiple reviewers: 11
Percentage of multi-reviewer PRs: 31.4%

📋 EXAMPLES OF MULTIPLE REVIEWER EVALUATION:

  Example 1: PR #6805
    Actual reviewers (3): bjohansebas, AkashDev01, tausiq2003
    Predicted top-3: UlisesGascon, wesleytodd, Phillip9587
    Matches found: None
    Hit@3: ❌ NO
    Precision@3: 0.000 (0/3)
    Recall@3: 0.000 (0/3)

  Example 2: PR #6810
    Actual reviewers (3): avivkeller, lululolily, Cycloctane
    Predicted top-3: wesleytodd, UlisesGascon, sheplu
    Matches found: None
    Hit@3: ❌ NO
    Precision@3: 0.000 (0/3)
    Recall@3: 0.000 (0/3)

  Example 3: PR #6699
    Actual reviewers (2): UlisesGascon, bjohansebas
    Predicted top-3: wesleytodd, bjohansebas, jonchurch
    Matches found: bjohansebas
    Hit

In [92]:
# ==================== SAVE METRICS TO JSON ====================
metrics_results = {
    'evaluation_metadata': {
        'total_prs_evaluated': len(common_prs),
        'unique_actual_reviewers': stability['unique_actual_reviewers'],
        'unique_predicted_reviewers': stability['unique_predicted_reviewers']
    },
    'precision_at_k': {
        'precision_at_1': precision_at_1,
        'precision_at_3': precision_at_3,
        'precision_at_5': precision_at_5,
        'precision_at_10': precision_at_10
    },
    'recall_at_k': {
        'recall_at_1': recall_at_1,
        'recall_at_3': recall_at_3,
        'recall_at_5': recall_at_5,
        'recall_at_10': recall_at_10
    },
    'f1_score_at_k': {
        'f1_at_1': f1_at_1,
        'f1_at_3': f1_at_3,
        'f1_at_5': f1_at_5,
        'f1_at_10': f1_at_10
    },
    'hit_rate_at_k': {
        'hit_at_1': hit_at_1,
        'hit_at_3': hit_at_3,
        'hit_at_5': hit_at_5,
        'hit_at_10': hit_at_10
    },
    'mean_reciprocal_rank': {
        'mrr': mrr,
        'mrr_percentage': mrr * 100
    },
    'mean_average_precision_at_k': {
        'map_at_1': map_at_1,
        'map_at_3': map_at_3,
        'map_at_5': map_at_5,
        'map_at_10': map_at_10
    },
    'dcg_at_k': {
        'dcg_at_1': dcg_at_1,
        'dcg_at_3': dcg_at_3,
        'dcg_at_5': dcg_at_5,
        'dcg_at_10': dcg_at_10
    },
    'ndcg_at_k': {
        'ndcg_at_1': ndcg_at_1,
        'ndcg_at_3': ndcg_at_3,
        'ndcg_at_5': ndcg_at_5,
        'ndcg_at_10': ndcg_at_10
    },
    'stability_metrics': {
        'coverage': stability['coverage'],
        'exact_match_rate': stability['exact_match_rate'],
        'diversity_ratio': stability['diversity_ratio']
    }
}

# Save metrics to JSON file
with open('pr_reviewer_evaluation_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_results, f, indent=2, ensure_ascii=False)

print("✅ Metrics saved to 'pr_reviewer_evaluation_metrics.json'")

✅ Metrics saved to 'pr_reviewer_evaluation_metrics.json'


## 6. Detailed Analysis

In [93]:
# Analyze correct and incorrect predictions with Top-K evaluation
correct_predictions = []
incorrect_predictions = []
partial_matches = []

for pr_num in sorted(common_prs):
    actual = set(actual_reviewers[pr_num])
    predicted_top3 = predicted_reviewers[pr_num][:3] if predicted_reviewers[pr_num] else []
    
    # Check how many of top-3 predictions match actual reviewers
    matching_predictions = [p for p in predicted_top3 if p in actual]
    
    if matching_predictions:
        # At least one prediction matches
        correct_predictions.append({
            'pr_number': pr_num,
            'actual': list(actual),
            'predicted_top3': predicted_top3,
            'matching_predictions': matching_predictions,
            'match_position': [i+1 for i, p in enumerate(predicted_top3) if p in actual],
            'num_matches': len(matching_predictions)
        })
    else:
        # No predictions match
        incorrect_predictions.append({
            'pr_number': pr_num,
            'actual': list(actual),
            'predicted_top3': predicted_top3,
            'matching_predictions': [],
            'num_matches': 0
        })

print("="*80)
print("📊 TOP-3 PREDICTION ANALYSIS")
print("="*80)
print(f"Total PRs Evaluated: {len(common_prs)}")
print(f"Successful Matched (At least 1 match in top-3): {len(correct_predictions)}")
print(f"Failed to Match (No matches in top-3): {len(incorrect_predictions)}")
print(f"\nTop-3 Hit Rate: {(len(correct_predictions) / len(common_prs)) * 100:.2f}%")

📊 TOP-3 PREDICTION ANALYSIS
Total PRs Evaluated: 35
Successful Matched (At least 1 match in top-3): 16
Failed to Match (No matches in top-3): 19

Top-3 Hit Rate: 45.71%


## 6b. Quantitative Failure Analysis

A **miss** is a PR where none of the top-3 suggested reviewers is among the people who actually reviewed it.

For each miss the cell below:

1. Maps the stored PR requirements (`technical_skills_needed`, `review_expertise_areas`, `review_type_needed`) onto the 17 knowledge-unit categories in `javascript_skill_matrix`.
2. Compares the top-1 suggestion with the strongest profiled actual reviewer on KU coverage, average matched-KU frequency, total reviews, reviews in the 90 days before the PR, experience, and hybrid score.
3. Counts how often the suggestion is stronger, then labels each miss **justified** (suggestion wins more metrics) or **genuine** (actual reviewer wins more metrics).

Hybrid score is the stored `match_score`. On a top-3 miss the actual reviewer was not scored, so that metric is left out of the vote. Recent reviews come from the full reviewer history, counted before the PR opened.

In [94]:
import json
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

BASE = Path(".")
PROFILE_DIR = BASE / ".." / "Reviewers Profiles"
REVIEW_HISTORY_PATH = BASE / ".." / "PR data for Reviewers" / "expressjs_express_reviewer_data.json"
RECENT_WINDOW = timedelta(days=90)

# Requirement text -> the 17 knowledge-unit categories used in reviewer profiles.
KU_KEYWORDS = {
    "Framework/Library Expertise": [
        "express", "node.js", "nodejs", "javascript", "supertest", "superagent",
        "mocha", "jest", "react", "koa",
    ],
    "Asynchronous Programming": ["async", "await", "promise", "concurrency", "event loop"],
    "API Design & Consumption": ["api", "restful", "request object", "http server", "endpoint"],
    "Error Handling": ["error", "exception", "try-catch", "deprecation"],
    "Frontend Development Skills": ["frontend", "dom ", "css", "html"],
    "Backend Development Skills": [
        "backend", "middleware", "server", "route", "http", "cors", "helmet", "compression",
    ],
    "DevOps & Deployment": [
        "github action", "ci/cd", "workflow", "setup-node", "actions/checkout", "codeql",
        "release", "deploy", "docker", "pipeline",
    ],
    "Advanced JavaScript Concepts": ["closure", "prototype", "module system", "memory"],
    "Functional Programming": ["functional", "immutab", "pure function"],
    "Data Structures & Algorithms": ["algorithm", "data structure"],
    "TypeScript": ["typescript"],
    "Progressive Web Apps (PWA)": ["service worker", "pwa"],
    "Mobile Development with JavaScript": ["react native", "ionic"],
    "Web Performance Optimization": ["performance", "caching", "profil", "lazy load"],
    "Event-Driven Architecture": ["event emitter", "pub/sub", "event-driven"],
    "Dependency Management": [
        "npm", "yarn", "dependabot", "dependency", "deps", "package management",
        "versioning", "cookie-session", "package-lock",
    ],
    "Graphical Data Visualization": ["d3.js", "chart.js", "webgl"],
}
EXP_RANK = {
    "junior": 1,
    "mid": 2,
    "mid-level": 2,
    "intermediate": 2,
    "senior": 3,
    "advanced": 3,
    "lead": 4,
}
VOTE_METRICS = [
    "ku_coverage",
    "avg_ku_frequency",
    "total_reviews",
    "recent_reviews",
    "hybrid_score",
]


def parse_time(value):
    if not value:
        return None
    text = value.replace("Z", "+00:00")
    try:
        parsed = datetime.fromisoformat(text)
    except ValueError:
        return None
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed


def load_profiles(profile_dir):
    profiles = {}
    for path in Path(profile_dir).glob("*.json"):
        with open(path, encoding="utf-8") as handle:
            doc = json.load(handle)
        name = (doc.get("reviewer_name") or "").strip()
        if name:
            profiles[name.lower()] = doc
    return profiles


def load_review_times(path):
    with open(path, encoding="utf-8") as handle:
        history = json.load(handle)
    by_reviewer = defaultdict(list)
    for pr in history.get("pr_reviewer_data", []):
        pr_number = pr.get("pr_number")
        for review in pr.get("reviews") or []:
            username = (review.get("reviewer_username") or "").strip().lower()
            submitted = parse_time(review.get("submitted_at"))
            if username and submitted:
                by_reviewer[username].append((submitted, pr_number))
    return by_reviewer


def parse_skill_matrix(doc):
    matrix = ((doc or {}).get("profile") or {}).get("javascript_skill_matrix") or {}
    categories = {}
    for key, skills in matrix.items():
        name = key.split(", frequency:")[0].strip()
        freq = 0
        if "frequency:" in key:
            try:
                freq = int(key.rsplit("frequency:", 1)[-1].strip())
            except ValueError:
                freq = 0
        child_total = 0
        if isinstance(skills, list):
            for skill in skills:
                if isinstance(skill, str) and "frequency:" in skill:
                    try:
                        child_total += int(skill.rsplit("frequency:", 1)[-1].strip())
                    except ValueError:
                        pass
        if freq <= 0 and child_total > 0:
            freq = child_total
        categories[name.lower()] = {"name": name, "frequency": max(freq, 0)}
    return categories


def required_kus(pr):
    analysis = pr.get("detailed_analysis") or {}
    texts = []
    for field in ("technical_skills_needed", "review_expertise_areas"):
        value = analysis.get(field) or []
        texts.extend(value if isinstance(value, list) else [value])
    if analysis.get("review_type_needed"):
        texts.append(analysis["review_type_needed"])
    found = []
    for category, keywords in KU_KEYWORDS.items():
        for item in texts:
            lowered = str(item).lower()
            if any(keyword in lowered for keyword in keywords):
                found.append(category)
                break
    return found


def coverage_and_frequency(doc, kus):
    categories = parse_skill_matrix(doc)
    matched_freqs = []
    for ku in kus:
        info = categories.get(ku.lower())
        if info and info["frequency"] > 0:
            matched_freqs.append(info["frequency"])
    covered = len(matched_freqs)
    avg_freq = sum(matched_freqs) / covered if covered else 0.0
    return covered, avg_freq


def recent_review_count(review_times, username, pr_created, pr_number):
    if pr_created is None:
        return None
    window_start = pr_created - RECENT_WINDOW
    return sum(
        1
        for submitted, reviewed_pr in review_times.get(username.lower(), [])
        if window_start <= submitted < pr_created and reviewed_pr != pr_number
    )


def experience_label(doc):
    label = ((doc or {}).get("profile") or {}).get("experience_level") or "Unknown"
    return str(label).strip()


def experience_rank(label):
    token = label.lower().split("(")[0].strip()
    return EXP_RANK.get(token)


def better(actual, suggested, higher_is_better=True):
    if actual is None or suggested is None:
        return None
    if actual == suggested:
        return "Tie"
    actual_wins = actual > suggested if higher_is_better else actual < suggested
    return "Actual" if actual_wins else "Suggested"


def top_reviewer(pr):
    ranked = pr.get("top_3_reviewers") or []
    if ranked:
        return ranked[0].get("reviewer_name"), ranked[0].get("match_score")
    return pr.get("assigned_reviewer"), pr.get("match_score")


def top_k_names(pr, k=3):
    ranked = pr.get("top_3_reviewers") or []
    names = [item.get("reviewer_name") for item in ranked if item.get("reviewer_name")]
    if not names and pr.get("assigned_reviewer"):
        names = [pr["assigned_reviewer"]]
    return names[:k]


def stored_score(pr, username):
    for ranked in pr.get("top_3_reviewers") or []:
        if (ranked.get("reviewer_name") or "").lower() == username.lower():
            try:
                return float(ranked.get("match_score"))
            except (TypeError, ValueError):
                return None
    return None


def is_bot(name):
    lowered = name.lower()
    return lowered.endswith("[bot]") or "bot" in lowered.split("-")


def choose_actual_reviewer(names, profiles, kus):
    humans = [name for name in names if name and not is_bot(name)]
    pool = humans or [name for name in names if name]
    profiled = [name for name in pool if name.lower() in profiles]
    if not profiled:
        return None

    def strength(name):
        covered, avg_freq = coverage_and_frequency(profiles[name.lower()], kus)
        total = int(((profiles[name.lower()].get("stats") or {}).get("total_reviews") or 0))
        return (covered, avg_freq, total)

    return max(profiled, key=strength)


def person_metrics(doc, kus, review_times, username, pr_created, pr_number, hybrid):
    covered, avg_freq = coverage_and_frequency(doc, kus)
    total = int(((doc or {}).get("stats") or {}).get("total_reviews") or 0)
    label = experience_label(doc)
    return {
        "ku_coverage": covered,
        "ku_required": len(kus),
        "avg_ku_frequency": round(avg_freq, 2),
        "total_reviews": total,
        "recent_reviews": recent_review_count(review_times, username, pr_created, pr_number),
        "experience_level": label,
        "experience_rank": experience_rank(label),
        "hybrid_score": None if hybrid is None else round(float(hybrid), 4),
    }


def classify_case(comparisons):
    suggested_wins = 0
    actual_wins = 0
    comparable = 0
    for metric in VOTE_METRICS:
        winner = comparisons[metric]
        if winner is None:
            continue
        comparable += 1
        if winner == "Suggested":
            suggested_wins += 1
        elif winner == "Actual":
            actual_wins += 1
    if comparable == 0 or suggested_wins == actual_wins:
        label = "Split"
    elif suggested_wins > actual_wins:
        label = "Justified"
    else:
        label = "Genuine"
    return label, suggested_wins, actual_wins, comparable


profiles = load_profiles(PROFILE_DIR)
review_times = load_review_times(REVIEW_HISTORY_PATH)
actual_by_pr = {pr["pr_number"]: pr for pr in actual_data["pr_reviewer_assignments"]}

miss_rows = []
unscored = []
for pr in predicted_data["pr_assignments"]:
    pr_number = pr["pr_number"]
    actual_pr = actual_by_pr.get(pr_number)
    if actual_pr is None:
        continue
    suggested_names = top_k_names(pr, k=3)
    suggested_name, suggested_hybrid = top_reviewer(pr)
    actual_names = actual_pr.get("unique_reviewers") or []
    actual_lower = {name.lower() for name in actual_names}
    if not suggested_names or any(name.lower() in actual_lower for name in suggested_names):
        continue

    kus = required_kus(pr)
    actual_name = choose_actual_reviewer(actual_names, profiles, kus)
    suggested_doc = profiles.get(suggested_name.lower())
    if actual_name is None or suggested_doc is None:
        unscored.append({
            "pr_number": pr_number,
            "pr_title": pr.get("pr_title"),
            "suggested_reviewer": suggested_name,
            "actual_reviewers": actual_names,
            "reason": "missing profile for the actual reviewer" if actual_name is None else "missing profile for the suggested reviewer",
        })
        continue

    pr_created = parse_time(actual_pr.get("created_at"))
    actual_metrics = person_metrics(
        profiles[actual_name.lower()], kus, review_times, actual_name, pr_created, pr_number,
        stored_score(pr, actual_name),
    )
    suggested_metrics = person_metrics(
        suggested_doc, kus, review_times, suggested_name, pr_created, pr_number, suggested_hybrid,
    )
    comparisons = {
        "ku_coverage": None if not kus else better(actual_metrics["ku_coverage"], suggested_metrics["ku_coverage"]),
        "avg_ku_frequency": None if not kus else better(actual_metrics["avg_ku_frequency"], suggested_metrics["avg_ku_frequency"]),
        "total_reviews": better(actual_metrics["total_reviews"], suggested_metrics["total_reviews"]),
        "recent_reviews": better(actual_metrics["recent_reviews"], suggested_metrics["recent_reviews"]),
        "experience": better(actual_metrics["experience_rank"], suggested_metrics["experience_rank"]),
        "hybrid_score": better(actual_metrics["hybrid_score"], suggested_metrics["hybrid_score"]),
    }
    label, suggested_wins, actual_wins, comparable = classify_case(comparisons)
    miss_rows.append({
        "pr_number": pr_number,
        "pr_title": pr.get("pr_title"),
        "required_kus": kus,
        "actual_reviewer": actual_name,
        "suggested_reviewer": suggested_name,
        "suggested_top3": suggested_names,
        "actual_reviewers": actual_names,
        "actual": actual_metrics,
        "suggested": suggested_metrics,
        "better": comparisons,
        "suggested_metric_wins": suggested_wins,
        "actual_metric_wins": actual_wins,
        "comparable_metrics": comparable,
        "classification": label,
    })


def pct(numerator, denominator):
    return round(100.0 * numerator / denominator, 1) if denominator else 0.0


def metric_summary(metric):
    decided = [row for row in miss_rows if row["better"][metric] is not None]
    suggested_better = sum(row["better"][metric] == "Suggested" for row in decided)
    actual_better = sum(row["better"][metric] == "Actual" for row in decided)
    ties = sum(row["better"][metric] == "Tie" for row in decided)
    return {
        "compared": len(decided),
        "suggested_better": suggested_better,
        "actual_better": actual_better,
        "ties": ties,
        "suggested_better_pct": pct(suggested_better, len(decided)),
    }


aggregate = {metric: metric_summary(metric) for metric in VOTE_METRICS}
class_counts = Counter(row["classification"] for row in miss_rows)
n_compared = len(miss_rows)
genuine_rows = [row for row in miss_rows if row["classification"] == "Genuine"]
genuine_kus = Counter(ku for row in genuine_rows for ku in row["required_kus"])
genuine_metric_wins = {
    metric: sum(row["better"][metric] == "Actual" for row in genuine_rows)
    for metric in VOTE_METRICS
}

labels = {
    "ku_coverage": "better KU coverage",
    "avg_ku_frequency": "higher KU frequency",
    "total_reviews": "more total review activity",
    "recent_reviews": "more recent activity",
    "hybrid_score": "a higher hybrid score",
}
print("=" * 88)
print("REVIEWER MISS FAILURE ANALYSIS")
print("=" * 88)
print(f"PRs evaluated: {len(actual_by_pr)}")
print(f"Top-3 misses: {n_compared + len(unscored)}")
print(f"Compared (both people have profiles): {n_compared}")
print(f"Unscored (profile missing): {len(unscored)}")
print()
print("Across compared misses, the suggested reviewer had:")
for metric, text in labels.items():
    stats = aggregate[metric]
    print(
        f"  - {text} in {stats['suggested_better_pct']}% "
        f"({stats['suggested_better']}/{stats['compared']}; "
        f"actual better {stats['actual_better']}, ties {stats['ties']})"
    )
if aggregate["hybrid_score"]["compared"] == 0:
    print(
        "Hybrid score is not compared on these misses: the actual reviewer was outside the scored top 3."
    )
else:
    print(
        "Hybrid score is compared only when the actual reviewer was also scored. "
        "On a top-3 miss that is rare, because the actual reviewer was outside the scored list."
    )

print()
print(
    f"Out of {n_compared} compared misses, "
    f"{pct(class_counts['Justified'], n_compared)}% were justified and "
    f"{pct(class_counts['Genuine'], n_compared)}% were genuine "
    f"({class_counts['Justified']} justified, {class_counts['Genuine']} genuine, "
    f"{class_counts['Split']} splits)."
)
if unscored:
    print(f"{len(unscored)} additional misses were not classified because a profile was missing.")

justified_share = pct(class_counts["Justified"], n_compared) / 100
genuine_share = pct(class_counts["Genuine"], n_compared) / 100
print()
print("Conclusion")
if justified_share >= 0.6:
    print(
        "Most compared misses are justified. The suggested reviewer is stronger on more "
        "of the measured metrics than the person who reviewed the PR, so a disagreement "
        "with the recorded reviewer does not by itself show that the recommendation is wrong."
    )
if genuine_share >= 0.3:
    print(
        "A substantial share of misses are genuine. On those PRs the actual reviewer was "
        "stronger on more metrics, so the ranker missed a better-supported person."
    )
    if genuine_kus:
        top_kus = ", ".join(f"{name} ({count})" for name, count in genuine_kus.most_common(5))
        print(f"Knowledge units that appear most often on genuine misses: {top_kus}.")
    failing = sorted(genuine_metric_wins.items(), key=lambda item: item[1], reverse=True)
    print(
        "On genuine misses, the actual reviewer was stronger most often on "
        + ", ".join(f"{labels[metric]} ({count})" for metric, count in failing if count)
        + "."
    )
elif justified_share < 0.6:
    print(
        "The misses are mixed. Justified cases support the recommender when it disagrees "
        "with the recorded reviewer; genuine cases are the ones to inspect by knowledge unit."
    )
    if genuine_kus:
        top_kus = ", ".join(f"{name} ({count})" for name, count in genuine_kus.most_common(5))
        print(f"Knowledge units that appear most often on genuine misses: {top_kus}.")

table = pd.DataFrame([
    {
        "PR": row["pr_number"],
        "Actual": row["actual_reviewer"],
        "Suggested": row["suggested_reviewer"],
        "Top 3": ", ".join(row["suggested_top3"]),
        "KUs": ", ".join(row["required_kus"]) if row["required_kus"] else "(none mapped)",
        "Coverage A": f"{row['actual']['ku_coverage']}/{row['actual']['ku_required']}",
        "Coverage S": f"{row['suggested']['ku_coverage']}/{row['suggested']['ku_required']}",
        "Cov": row["better"]["ku_coverage"] or "n/a",
        "Freq A": row["actual"]["avg_ku_frequency"],
        "Freq S": row["suggested"]["avg_ku_frequency"],
        "Freq": row["better"]["avg_ku_frequency"] or "n/a",
        "Reviews A": row["actual"]["total_reviews"],
        "Reviews S": row["suggested"]["total_reviews"],
        "Rev": row["better"]["total_reviews"],
        "Recent A": row["actual"]["recent_reviews"],
        "Recent S": row["suggested"]["recent_reviews"],
        "Rec": row["better"]["recent_reviews"] or "n/a",
        "Level A": row["actual"]["experience_level"],
        "Level S": row["suggested"]["experience_level"],
        "Hybrid A": row["actual"]["hybrid_score"] if row["actual"]["hybrid_score"] is not None else "not scored",
        "Hybrid S": row["suggested"]["hybrid_score"],
        "Hyb": row["better"]["hybrid_score"] or "n/a",
        "Class": row["classification"],
    }
    for row in miss_rows
])
print()
print(table.to_string(index=False))

failure_result = {
    "definition": "none of the top-3 suggested reviewers is among the PR's actual reviewers",
    "comparison": "each miss is compared with the top-1 suggestion",
    "recent_window_days": 90,
    "prs_evaluated": len(actual_by_pr),
    "top1_misses": n_compared + len(unscored),
    "compared_misses": n_compared,
    "unscored_misses": unscored,
    "aggregate": aggregate,
    "classification": {
        "justified": class_counts["Justified"],
        "genuine": class_counts["Genuine"],
        "split": class_counts["Split"],
        "justified_pct": pct(class_counts["Justified"], n_compared),
        "genuine_pct": pct(class_counts["Genuine"], n_compared),
        "split_pct": pct(class_counts["Split"], n_compared),
    },
    "genuine_ku_counts": dict(genuine_kus),
    "genuine_metric_wins_for_actual": genuine_metric_wins,
    "misses": miss_rows,
}
out_path = BASE / "reviewer_miss_failure_analysis.json"
with open(out_path, "w", encoding="utf-8") as handle:
    json.dump(failure_result, handle, indent=2)
print(f"\nSaved: {out_path}")

REVIEWER MISS FAILURE ANALYSIS
PRs evaluated: 35
Top-3 misses: 19
Compared (both people have profiles): 13
Unscored (profile missing): 6

Across compared misses, the suggested reviewer had:
  - better KU coverage in 33.3% (3/9; actual better 0, ties 6)
  - higher KU frequency in 88.9% (8/9; actual better 1, ties 0)
  - more total review activity in 69.2% (9/13; actual better 4, ties 0)
  - more recent activity in 23.1% (3/13; actual better 8, ties 2)
  - a higher hybrid score in 0.0% (0/0; actual better 0, ties 0)
Hybrid score is not compared on these misses: the actual reviewer was outside the scored top 3.

Out of 13 compared misses, 53.8% were justified and 15.4% were genuine (7 justified, 2 genuine, 4 splits).
6 additional misses were not classified because a profile was missing.

Conclusion
The misses are mixed. Justified cases support the recommender when it disagrees with the recorded reviewer; genuine cases are the ones to inspect by knowledge unit.
Knowledge units that appear 

## 7. LLM-based Failure Analysis

In [56]:
import os

def get_pr_details(pr_num):
    """Extract PR details from the loaded data"""
    # Get PR details from predicted data (which has more info)
    for pr in predicted_data['pr_assignments']:
        if pr['pr_number'] == pr_num:
            return {
                'title': pr.get('pr_title', 'N/A'),
                'description': pr.get('pr_requirements', {}).get('description', 'N/A')[:300],  # Limit length
                'labels': pr.get('pr_requirements', {}).get('labels', []),
                'files_changed': pr.get('pr_requirements', {}).get('files_changed', 'N/A'),
                'complexity': pr.get('pr_requirements', {}).get('complexity', 'N/A'),
                'top_3_scores': [
                    f"{r['reviewer_name']} (score: {r.get('match_score', 'N/A')})" 
                    for r in pr.get('top_3_reviewers', [])
                ]
            }
    return None

def analyze_failure_with_llm(failure_info):
    """Use LLM to analyze why a reviewer assignment failed with actual PR context"""
    
    pr_num = failure_info['pr_number']
    actual = failure_info['actual']
    predicted = failure_info['predicted_top3']
    
    # Get actual PR details
    pr_details = get_pr_details(pr_num)
    
    if pr_details:
        prompt = f"""Analyze why this PR reviewer assignment failed to match:

**PR #{pr_num}**
Title: {pr_details['title']}
Description: {pr_details['description']}
Labels: {', '.join(pr_details['labels']) if pr_details['labels'] else 'None'}
Files Changed: {pr_details['files_changed']}
Complexity: {pr_details['complexity']}

**Actual Reviewers Assigned:** {', '.join(actual)}
**Predicted Top 3:** {', '.join(predicted)}

**Top 3 Predictions with Scores:**
{chr(10).join(pr_details['top_3_scores'])}

Based on the PR details above, Provide a concise reason why the actual reviewer was better 
suited than the predicted ones. Focus on the most important factor only. Check the PR properly and avoid generic answers. Also distinguish betwwen the actual and predicted reviewers."""
    
    # Configure environment variables
    os.environ["OPENAI_API_KEY"] = "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"
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

    llm = ChatOpenAI(
        model="mistralai/ministral-8b-2512",
        temperature=0.1,
        max_tokens=4000,
        openai_api_key=os.environ.get("OPENAI_API_KEY"),
        openai_api_base=os.environ.get("OPENAI_API_BASE")
    )

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
print("🤖 LLM-BASED FAILURE ANALYSIS")

if not incorrect_predictions:
    print("\n✅ All matched!")
else:
    # Analyze all failed predictions
    num_to_analyze = len(incorrect_predictions)
    
    for i in range(num_to_analyze):
        failure = incorrect_predictions[i]
        
        print(f"\n📋 Failure #{i+1}")
        print(f"   PR Number: {failure['pr_number']}")
        print(f"   Actual: {', '.join(failure['actual'])}")
        print(f"   Predicted: {', '.join(failure['predicted_top3'])}")
        print(f"\n   🧠 LLM Analysis:")
        
        analysis = analyze_failure_with_llm(failure)
        
        # Format output
        for line in analysis.split('\n'):
            if line.strip():
                print(f"      {line.strip()}")

🤖 LLM-BASED FAILURE ANALYSIS

📋 Failure #1
   PR Number: 6703
   Actual: hamidrezaghavami
   Predicted: bjohansebas, wesleytodd, sheplu

   🧠 LLM Analysis:
      The actual reviewer, **hamidrezaghavami**, was better suited for this PR compared to the predicted reviewers (bjohansebas, wesleytodd, sheplu) because **hamidrezaghavami** has a history of contributing to and reviewing test-related changes in the codebase, particularly in the area of request validation and testing. This specific PR focuses on enhancing `req.is()` tests, and **hamidrezaghavami**'s expertise and recent activity in this domain make them a more appropriate choice for reviewing this PR. The predicted reviewers, while generally active, may not have the same level of recent involvement or specific expertise in this particular area of the codebase.

📋 Failure #2
   PR Number: 6705
   Actual: Phillip9587, UlisesGascon
   Predicted: wesleytodd, bjohansebas, rileyjshaw

   🧠 LLM Analysis:
      The actual reviewers, Phil